# Notebook 07: The core result: the Mitophagy-Lysosome Coupling Index (MLCI) across age

**Goal.** One MLCI per mouse, a meta-analysis across cohorts, a random-gene-set null and four negative-control pairs. This notebook tests hypotheses H1 and H2.

MLCI = partial Spearman correlation between the mitophagy score and the lysosome score *across the cells of one mouse*, after removing the effect of nuisance variables. Three levels: **M0** raw; **M1** adjusted for depth and mitochondrial %; **M2** (primary) also adjusted for stress score and cell state. Report all three: they show how much each confounder matters.

**Cell 1.** Loads the tools and the shared settings. **sys.path.insert** tells Python where the project code lives; **mlc.config** gives every notebook the same folders (PROC = processed data, RES = results tables, FIG = figures) and the profile (*sample* = built-in simulated data, *real* = your downloads). It also tells pandas to print wide tables on one screen. This cell is identical in every notebook.

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from mlc.config import *
from mlc import genesets as G
pd.set_option("display.width", 150); pd.set_option("display.max_columns", 20)
print("profile:", PROFILE, "| cohorts:", cohorts())

**Cell 2.** **The central computation.** For every mouse: take its cells, regress the nuisance variables out of the ranked scores, correlate the residuals (r), convert to Fisher z (a scale suited to averaging), and bootstrap the cells 100 times for a 95% interval. **depth** = log total UMIs; **pct_mt** = mitochondrial %; **dissoc** = stress score; **state_*** = cell-state indicators.

*You should see:* Average MLCI (r) falling with age: about 0.39, 0.32, 0.19 at 3, 12, 24 months (SIM_A) and 0.28 to 0.13 (SIM_B).

In [ ]:
from mlc.dataset import Dataset
from mlc.coupling import *
from mlc import plots
sheet = pd.read_csv(SHEET).set_index("sample_id")
obsd = {c: Dataset.load(PROC / f"{c}_scored").obs for c in cohorts()["mouse"]}
levels = {"M0": [], "M1": ["depth", "pct_mt"], "M2": ["depth", "pct_mt", "dissoc", "STATE"]}
rows = []
for c, o in obsd.items():
    state_cols = [x for x in o.columns if x.startswith("state_")]
    for lv, cov in levels.items():
        cv = [x for x in cov if x != "STATE"] + (state_cols if "STATE" in cov else [])
        r = coupling_per_sample(o, "m_mitophagy", "m_lysosome", covars=cv, min_cells=MIN_CELLS, n_boot=100, seed=SEED)
        r["level"], r["cohort"] = lv, c; rows.append(r)
R = pd.concat(rows, ignore_index=True)
R["age_months"] = R.sample_id.map(sheet.age_months).astype(float); R["log2_age"] = np.log2(R.age_months)
R.to_csv(RES / "mlci_per_sample.csv", index=False)
print(R[R.level == "M2"].groupby(["cohort", "age_months"]).r.mean().round(2))

**Cell 3.** **Age effect.** Within each cohort, a straight-line fit of MLCI (z) against log2(age) gives a slope (change in z each time age doubles) with its standard error. Slopes are then pooled across cohorts by a **random-effects meta-analysis**. A negative pooled slope with a confidence interval below 0 supports H1. **I2** = percentage of variation between cohorts that is real heterogeneity; with only 2-3 cohorts treat it as rough.

*You should see:* Slope about -0.07 in each cohort; pooled M2 slope -0.067 (95% interval about -0.088 to -0.047), p far below 0.05. M0, M1 and M2 almost identical here; in real data they usually differ more.

In [ ]:
S = cohort_slopes(R)
pooled = {lv: dersimonian_laird(g.beta, g.se) for lv, g in S.groupby("level")}
P_ = pd.DataFrame(pooled).T
print(S.round(3)); print(P_[["beta", "ci_lo", "ci_hi", "p", "I2", "k"]].round(4))
S.to_csv(RES / "mlci_cohort_slopes.csv", index=False); P_.to_csv(RES / "mlci_meta.csv")

**Cell 4.** **The matched random null.** Is a decline this steep special to *these* gene sets, or would any two gene sets of the same size and expression level decline like this? Each permutation draws a random set for each arm (same size, same expression-level profile as your real sets), recomputes the MLCI in every mouse and the age slope. The permutation p is the fraction of random pairs with a slope at least as extreme as yours. The smallest possible p is 1/(n_perm+1) = 0.005 here.

*You should see:* Observed slope about -0.07; null spread (sd) about 0.011-0.014; permutation p = 0.005 (the smallest value possible with 200 permutations).

In [ ]:
from mlc import genesets as G
n_perm = 200                      # use 1000+ for real analyses (takes longer)
nulls, obs_slopes = {}, {}
for c in cohorts()["mouse"]:
    ds = Dataset.load(PROC / f"{c}_scored")
    keep = np.asarray((ds.L > 0).mean(axis=0)).ravel() > 0.05             # genes seen in >5% of cells
    ds = ds.subset_genes(keep); means = np.asarray(ds.L.mean(axis=0)).ravel()
    iA, iB = ds.gene_idx(G.MITOPHAGY), ds.gene_idx(G.LYSOSOME)
    ids = R[(R.cohort == c) & (R.level == "M0")].sample_id.tolist()
    Xs = {s: ds.L[(ds.obs.sample_id == s).values].toarray() for s in ids}
    ages = np.log2(sheet.loc[ids, "age_months"].astype(float).values); coh = np.array([c] * len(ids))
    ca, cb = matched_controls(means, iA), matched_controls(means, iB)
    z = [np.arctanh(partial_spearman(fast_score(Xs[s], iA, ca), fast_score(Xs[s], iB, cb))) for s in ids]
    obs_slopes[c] = age_slope(z, ages, coh)
    nulls[c] = perm_null_age_slope(Xs, means, iA, iB, ages, coh, n_perm=n_perm, seed=SEED)
    p = (1 + (np.abs(nulls[c]) >= abs(obs_slopes[c])).sum()) / (n_perm + 1)
    print(c, "observed slope", round(obs_slopes[c], 3), "| null sd", round(nulls[c].std(), 3),
          "| permutation p", round(p, 4))
pooled_null = np.mean(list(nulls.values()), axis=0); pooled_obs = np.mean(list(obs_slopes.values()))
p_pool = (1 + (np.abs(pooled_null) >= abs(pooled_obs)).sum()) / (n_perm + 1)
print("pooled observed", round(pooled_obs, 3), "permutation p", round(p_pool, 4))
pd.DataFrame({"null": pooled_null}).to_csv(RES / "perm_null_pooled.csv", index=False)

**Cell 5.** **Specificity (H2).** Repeats the age-slope for four negative-control pairs (ribosome-lysosome, proteasome-lysosome, oxidative-phosphorylation-mitophagy, core autophagy-lysosome). If the mitophagy-lysosome slope is clearly more negative than all of these, the effect is specific; if the control pairs decline just as much, the finding is a general loss of co-expression, not coupling.

*You should see:* Mitophagy-lysosome row about -0.07; control rows between -0.02 and +0.02.

In [ ]:
pairs = [("m_mitophagy", "m_lysosome"), ("m_ribosome", "m_lysosome"), ("m_proteasome", "m_lysosome"),
         ("m_oxphos", "m_mitophagy"), ("m_autophagy_core", "m_lysosome")]
rows = []
for c, o in obsd.items():
    cv = ["depth", "pct_mt", "dissoc"] + [x for x in o.columns if x.startswith("state_")]
    for a, b in pairs:
        r = coupling_per_sample(o, a, b, covars=cv, min_cells=MIN_CELLS, n_boot=0)
        r["pair"], r["cohort"] = f"{a[2:]}-{b[2:]}", c; rows.append(r)
SP = pd.concat(rows); SP["log2_age"] = np.log2(SP.sample_id.map(sheet.age_months).astype(float))
spec = cohort_slopes(SP, by=("pair", "cohort")); spec.to_csv(RES / "specificity_slopes.csv", index=False)
print(spec.pivot(index="pair", columns="cohort", values="beta").round(3))

**Cell 6.** **Figure 2.** (a) One point per mouse with its bootstrap interval and a fitted line per cohort. (b) Forest plot: cohort slopes and the pooled estimate (red) with 95% intervals. (c) Your observed slope (red line) against the matched-random-set null. (d) The mitophagy-lysosome pair (red) against the negative controls.

In [ ]:
fig = plots.fig_mlci(R, {"slopes": S, "pooled": pooled["M2"]}, pooled_null, pooled_obs, spec)
plt.show()